# The Seven Layers of AI: Layer 7: Artificial General Intelligence (AGI)

This notebook contains every example program from **Chapter 8** of the study guide *The Seven Layers of AI*, in the order they appear. Each program comes with the explanation that introduces it in the guide, the output printed in the guide, what to notice in that output, and the Investigate tasks.

**How to use it:** run the cells from top to bottom. Each program is self-contained, so you can also run any single program on its own after the setup cell. Change the code freely: the Investigate tasks suggest where to start.

## Programs in this notebook

1. `l7_contamination.py`: benchmark contamination
2. `l7_arc_search.py`: solving ARC-style puzzles by program search
3. `l7_distribution_shift.py`: the limits of narrow generalisation
4. `l7_forgetting.py`: catastrophic forgetting
5. `l7_forecasting.py`: why forecasts disagree
6. `l7_specification_gaming.py`: specification gaming

## Setup

The programs in this notebook use: `numpy scikit-learn`. If any are missing, uncomment the line below and run it once.

In [1]:
# %pip install numpy scikit-learn
%matplotlib inline

## 1. Benchmark contamination

**File:** `l7_contamination.py`

In [2]:
"""
Layer 7 example: why benchmark scores can overstate general ability.
 
File: l7_contamination.py
 
Purpose:
  If test questions (or close copies) appear in a model's training data, the
  model can score highly by RECALL rather than REASONING. This is called
  benchmark contamination, and it is one reason a high score on a famous test
  is weak evidence of general intelligence. The program simulates two
  'models' answering simple arithmetic word problems:
    memoriser : has seen part of the benchmark online and recalls answers
    reasoner  : actually computes the answer, but makes occasional slips
 
How it works:
  1. make_problem() writes a word problem and its correct answer.
  2. 200 problems form the "published" benchmark; 150 of them (75%) are
     marked as leaked, meaning the memoriser saw them during training.
  3. Both models are scored on the published benchmark and on 200 FRESH
     problems with the same structure but new numbers.
 
What to look for in the output:
  - On the published benchmark the memoriser (0.76) looks almost as good as
    the reasoner (0.93).
  - On fresh variants the memoriser collapses to 0.01, while the reasoner
    holds steady at 0.91. Only held-out tests reveal the difference.
 
NOTE: no real language model runs here. The two 'models' are simple Python
functions designed to behave like the two extremes.
 
Requires: Python 3.8+ standard library only (random).
"""
 
import random
 
# Fixed seed so the simulated benchmark and scores are the same every run.
random.seed(2)
 
 
def make_problem(a, b):
    # Returns (question text, correct answer) for one word problem.
    return f"A library has {a} books and buys {b} more. How many books now?", a + b
 
 
# The published benchmark: 200 problems with random two-digit numbers.
published = [make_problem(random.randint(10, 99), random.randint(10, 99))
             for _ in range(200)]
# 75% of the questions leaked into training data (for example, posted online).
leaked = set(q for q, _ in random.sample(published, 150))
# Maps each published question to its answer, for the memoriser's "recall".
answer_key = dict(published)
 
 
def memoriser(question):
    # Perfect recall of questions it has seen; otherwise a plausible guess.
    if question in leaked:
        return answer_key[question]           # perfect recall of seen items
    return random.randint(20, 198)             # otherwise a plausible-looking guess
 
 
def reasoner(question):
    # Reads the two numbers from the text and adds them, the way a person would.
    nums = [int(t) for t in question.split() if t.isdigit()]
    ans = nums[0] + nums[1]
    # 8% of the time it makes a small slip, like a careless human.
    return ans if random.random() > 0.08 else ans + random.choice([-10, -1, 1, 10])
 
 
def score(model, problems):
    # Proportion of problems the model answers exactly right.
    return sum(model(q) == a for q, a in problems) / len(problems)
 
 
# Fresh variants: same kind of question, new numbers, never seen by anyone.
fresh = [make_problem(random.randint(10, 99), random.randint(10, 99))
         for _ in range(200)]
print(f"{'model':10s} {'published benchmark':>20s} {'fresh variants':>15s}")
for name, m in [("memoriser", memoriser), ("reasoner", reasoner)]:
    print(f"{name:10s} {score(m, published):20.2f} {score(m, fresh):15.2f}")
print("\nOn the published benchmark the memoriser looks nearly as"
      " capable as the reasoner.")
print("Fresh, held-out problems reveal the difference. This is why"
      " benchmark designers keep")
print("private test sets and keep creating new tests, and why a"
      " high score is not proof of")
print("general intelligence.")

model       published benchmark  fresh variants
memoriser                  0.76            0.01
reasoner                   0.93            0.91

On the published benchmark the memoriser looks nearly as capable as the reasoner.
Fresh, held-out problems reveal the difference. This is why benchmark designers keep
private test sets and keep creating new tests, and why a high score is not proof of
general intelligence.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
model       published benchmark  fresh variants
memoriser                  0.76            0.01
reasoner                   0.93            0.91
 
On the published benchmark the memoriser looks nearly as capable as the reasoner.
Fresh, held-out problems reveal the difference. This is why benchmark designers keep
private test sets and keep creating new tests, and why a high score is not proof of
general intelligence.
```

**What the output shows**

Both simulated models look capable on the published test, but only the reasoner holds up on newly written problems. Real contamination is rarely this stark, which is exactly what makes it difficult to detect from benchmark scores alone.

**Investigate**

1. Reduce the leaked proportion from 75 percent to 20 percent. How much does the memoriser's published score change?
2. Create "paraphrased" variants that keep the same numbers but reword the question. Modify the memoriser so it recalls answers when the numbers match. What does this reveal about how contamination can hide?

## 2. Solving ARC-style puzzles by program search

**File:** `l7_arc_search.py`

In [3]:
"""
Layer 7 example: abstraction and reasoning puzzles, in the style of ARC
(Chollet, 2019), solved by PROGRAM SEARCH.
 
File: l7_arc_search.py
 
Purpose:
  Each puzzle gives 2 or 3 example input grids with their output grids. The
  solver must discover the transformation and apply it to a new test grid.
  A person usually spots the rule from a couple of examples. This program
  shows one way a machine can do the same, and where that way breaks down.
 
How it works:
  1. A small library of grid operations (flip, rotate, transpose, recolour,
     gravity, fill holes) forms a 'domain-specific language' (DSL).
  2. search() tries every sequence of 1, 2 or 3 operations, in order, and
     returns the first program that turns EVERY example input into its output.
  3. The program found is applied to the unseen test grid and checked.
  4. The last puzzle needs counting, which the DSL cannot express, so no
     program can be found however long the search runs.
 
What to look for in the output:
  - Three puzzles are solved correctly from two or three examples, after
    trying 12, 45 and 7 candidate programs.
  - The counting puzzle fails after all 399 candidates (7 + 49 + 343).
  - With 30 primitives and programs of length 5 there would be 24,300,000
    candidates: blind search does not scale, so a general learner needs
    prior knowledge to guide it.
 
Requires: NumPy (pip install numpy).
"""
 
import itertools
import numpy as np
 
 
# ---------------- primitive operations (the DSL) ------------------------------
# Each takes a grid (a 2-D NumPy array of colour numbers) and returns a new grid.
# 0 means an empty cell; 1, 2 and 3 are colours.
def flip_h(g):       return g[:, ::-1]           # mirror left-right
def flip_v(g):       return g[::-1, :]           # mirror top-bottom
def rotate(g):       return np.rot90(g, -1)      # rotate 90 degrees clockwise
def transpose(g):    return g.T                  # swap rows and columns
def recolour(g):     # swap colours 1 and 2
    out = g.copy(); out[g == 1] = 2; out[g == 2] = 1; return out
def gravity(g):      # non-zero cells fall to the bottom of each column
    out = np.zeros_like(g)
    for c in range(g.shape[1]):
        col = g[:, c][g[:, c] != 0]
        if len(col):
            out[-len(col):, c] = col
    return out
def fill_holes(g):   # zero cells surrounded on 4 sides by non-zero become 3
    out = g.copy()
    for r in range(1, g.shape[0] - 1):
        for c in range(1, g.shape[1] - 1):
            if g[r, c] == 0 and all(g[r+dr, c+dc]
                                    for dr, dc in [(1,0),(-1,0),(0,1),(0,-1)]):
                out[r, c] = 3
    return out
def identity(g):     return g                    # defined but not in the search
 
# The building blocks the search may combine: 7 operations.
PRIMITIVES = [flip_h, flip_v, rotate, transpose, recolour, gravity, fill_holes]
 
 
def search(examples, max_depth=3):
    """Try every program of length 1..max_depth; return the first fitting ALL."""
    tried = 0
    for depth in range(1, max_depth + 1):
        # itertools.product lists every ordered sequence of `depth` operations.
        for program in itertools.product(PRIMITIVES, repeat=depth):
            tried += 1
            # run() applies the operations in the program one after another.
            def run(g, p=program):
                for f in p:
                    g = f(g)
                return g
            # A program is accepted only if it explains every example exactly.
            if all(np.array_equal(run(np.array(i)), np.array(o)) for i, o in examples):
                return program, run, tried
    return None, None, tried
 
 
G = np.array
# Each puzzle: training pairs (input, output) and one test pair (input, answer).
PUZZLES = {
    "mirror then swap colours": dict(
        train=[(G([[1,0,0],[1,2,0],[0,0,0]]), G([[0,0,2],[0,1,2],[0,0,0]])),
               (G([[2,2,0],[0,1,0],[0,0,1]]), G([[0,1,1],[0,2,0],[2,0,0]]))],
        test=(G([[1,1,2],[0,0,2],[1,0,0]]), G([[1,2,2],[1,0,0],[0,0,2]]))),
    "things fall down, then rotate": dict(
        train=[(G([[1,0,2],[0,0,0],[0,1,0]]), G([[1,0,0],[1,0,0],[2,0,0]])),
               (G([[0,2,0],[1,0,0],[0,0,0]]), G([[1,0,0],[2,0,0],[0,0,0]])),
               (G([[2,2,2],[0,0,0],[0,0,0]]), G([[2,0,0],[2,0,0],[2,0,0]]))],
        test=(G([[0,0,1],[2,0,0],[0,1,0]]), G([[2,0,0],[1,0,0],[1,0,0]]))),
    "fill enclosed holes": dict(
        train=[(G([[0,1,0],[1,0,1],[0,1,0]]), G([[0,1,0],[1,3,1],[0,1,0]])),
               (G([[1,1,1,1],[1,0,0,1],[1,1,1,1]]),
                G([[1,1,1,1],[1,0,0,1],[1,1,1,1]]))],
        test=(G([[2,2,2],[2,0,2],[2,2,2]]), G([[2,2,2],[2,3,2],[2,2,2]]))),
    # The output is a bar whose length is the number of coloured cells.
    # No combination of the primitives can count, so this one must fail.
    "count cells -> bar (outside the DSL)": dict(
        train=[(G([[1,0],[0,1]]), G([[1,1]])),
               (G([[1,1],[1,0]]), G([[1,1,1]]))],
        test=(G([[1,1],[1,1]]), G([[1,1,1,1]]))),
}
 
for name, pz in PUZZLES.items():
    program, run, tried = search(pz["train"])
    print(f"PUZZLE: {name}")
    if program is None:
        print(f"  no program found after trying {tried} candidates"
              f" (the rule is not expressible)")
    else:
        print(f"  found {' -> '.join(f.__name__ for f in program)} after"
              f" {tried} candidates")
        # Generalisation test: apply the discovered program to the unseen grid.
        test_in, expected = pz["test"]
        pred = run(test_in)
        verdict = "no answer key" if expected is None else (
            "CORRECT" if np.array_equal(pred, expected) else "WRONG")
        print(f"  test prediction {pred.tolist()} ({verdict})")
    print()
 
# The size of the search space grows as (number of primitives) ** (length).
print(f"Candidate programs of length 1-3: {sum(len(PRIMITIVES)**d for d in (1,2,3))}.")
print(f"With 30 primitives and length 5: {30**5:,}. Blind search"
      f" explodes combinatorially,")
print("so efficient learners must GUIDE the search with prior knowledge and intuition.")

PUZZLE: mirror then swap colours
  found flip_h -> recolour after 12 candidates
  test prediction [[1, 2, 2], [1, 0, 0], [0, 0, 2]] (CORRECT)

PUZZLE: things fall down, then rotate
  found gravity -> rotate after 45 candidates
  test prediction [[2, 0, 0], [1, 0, 0], [1, 0, 0]] (CORRECT)

PUZZLE: fill enclosed holes
  found fill_holes after 7 candidates
  test prediction [[2, 2, 2], [2, 3, 2], [2, 2, 2]] (CORRECT)

PUZZLE: count cells -> bar (outside the DSL)
  no program found after trying 399 candidates (the rule is not expressible)

Candidate programs of length 1-3: 399.
With 30 primitives and length 5: 24,300,000. Blind search explodes combinatorially,
so efficient learners must GUIDE the search with prior knowledge and intuition.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
PUZZLE: mirror then swap colours
  found flip_h -> recolour after 12 candidates
  test prediction [[1, 2, 2], [1, 0, 0], [0, 0, 2]] (CORRECT)
 
PUZZLE: things fall down, then rotate
  found gravity -> rotate after 45 candidates
  test prediction [[2, 0, 0], [1, 0, 0], [1, 0, 0]] (CORRECT)
 
PUZZLE: fill enclosed holes
  found fill_holes after 7 candidates
  test prediction [[2, 2, 2], [2, 3, 2], [2, 2, 2]] (CORRECT)
 
PUZZLE: count cells -> bar (outside the DSL)
  no program found after trying 399 candidates (the rule is not expressible)
 
Candidate programs of length 1-3: 399.
With 30 primitives and length 5: 24,300,000. Blind search explodes combinatorially,
so efficient learners must GUIDE the search with prior knowledge and intuition.
```

**What the output shows**

The solver finds correct programs from just two or three examples when the rule can be built from its primitives, and fails completely when it cannot, however many candidates it tries. The earliest competitive ARC solvers worked in exactly this way, using brute-force search over hand-crafted primitives (ARC Prize Foundation, 2026b). The exercise shows two things that matter for AGI: generalising from few examples requires strong built-in priors, and blind search becomes impossible as the space of programs grows. Humans appear to combine rich prior knowledge with intuition that guides search, and replicating that combination is an open problem (Chollet, 2019; Lake et al., 2017).

**Investigate**

1. Add a primitive that crops a grid to the smallest rectangle containing non-zero cells, then design a puzzle that requires it.
2. Add the primitives needed to solve the "count cells" puzzle. Did adding them feel like teaching the solver to reason, or giving it the answer?
3. Try some real public ARC tasks at arcprize.org yourself and time how long each takes. Which kinds of rule would be hardest to express as primitives?

## 3. The limits of narrow generalisation

**File:** `l7_distribution_shift.py`

In [4]:
"""
Layer 7 example: how narrow a 'narrow' AI really is.
 
File: l7_distribution_shift.py
 
Purpose:
  A classifier trained on handwritten digits scores very well on new images
  drawn from the SAME distribution as its training data. This program changes
  the conditions in ways a person would barely notice, and then asks a network
  to extrapolate a simple rule (squaring) beyond the numbers it was shown.
  Both experiments test the difference between fitting examples and grasping
  a concept, which is at the heart of the question "how general is it?".
 
How it works:
  1. A small neural network (scikit-learn MLPClassifier, Chapter 4) is trained
     on 1,300 of the 8 x 8 digit images and tested on the remaining 497.
  2. The test images are transformed: inverted, mirrored, rotated, brightened
     and made noisy. The unchanged model is scored on each version.
  3. A second network learns y = x squared from 400 examples with x between
     -3 and 3, and is then asked about x = 5, 10 and 20.
 
What to look for in the output:
  - 0.984 on new images from the same distribution, but exactly 0.000 when
    black and white are swapped, a change that would not trouble any reader.
  - Brighter strokes and light noise barely matter (0.972 and 0.958): the
    model is robust to changes resembling its training data.
  - The squaring network is accurate inside -3 to 3 but predicts 73.5 for
    20 squared (true answer 400).
 
Requires: NumPy and scikit-learn (pip install numpy scikit-learn).
"""
 
import numpy as np
import warnings
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier, MLPRegressor
warnings.filterwarnings("ignore")      # hide convergence warnings for clarity
 
# 1,797 digit images, each 8 x 8 pixels with values 0 to 16, scaled to 0-1.
X, y = load_digits(return_X_y=True)
X = X / 16.0
rng = np.random.default_rng(0)
idx = rng.permutation(len(X))
tr, te = idx[:1300], idx[1300:]        # training and test indices
model = MLPClassifier(hidden_layer_sizes=(128,), max_iter=500,
                      random_state=0).fit(X[tr], y[tr])
 
# Reshape test rows back into 8 x 8 images so they can be flipped and rotated.
imgs = X[te].reshape(-1, 8, 8)
tests = {
    "same distribution (new images)": imgs,
    "colours inverted (white on black)": 1 - imgs,
    "mirrored left-right": imgs[:, :, ::-1],
    "rotated 90 degrees": np.rot90(imgs, 1, axes=(1, 2)),
    "thicker strokes (brighter)": np.clip(imgs * 2.0, 0, 1),
    "light noise added": np.clip(imgs + rng.normal(0, 0.15, imgs.shape), 0, 1),
}
print("Digit classifier accuracy (chance = 0.10):")
for name, data in tests.items():
    # Flatten each image back to 64 numbers, the format the model expects.
    print(f"  {name:36s} {model.score(data.reshape(len(data), -1), y[te]):.3f}")
print("(Mirroring and rotation genuinely change some digits, e.g. 6 and 9, so a human")
print(" would also hesitate; inversion and brightness should not"
      " confuse a reader at all.)\n")
 
# ---- Extrapolation: learning y = x squared ---------------------------------------
# Training inputs come only from the range -3 to 3.
x_train = rng.uniform(-3, 3, 400).reshape(-1, 1)
net = MLPRegressor(hidden_layer_sizes=(64, 64), max_iter=3000,
                   random_state=0).fit(x_train, x_train.ravel() ** 2)
print("A network learns y = x^2 from examples between -3 and 3:")
print("   x     true y   network")
# The first three x values are inside the training range; the last three are not.
for x in (-2.0, 0.5, 2.5, 5.0, 10.0, 20.0):
    print(f"{x:5.1f} {x*x:9.1f} {net.predict([[x]])[0]:9.1f}")
print("Inside the training range it is accurate; outside it the"
      " 'rule' was never learned.")
print("A student who understands squaring extrapolates immediately.")

Digit classifier accuracy (chance = 0.10):
  same distribution (new images)       0.984
  colours inverted (white on black)    0.000
  mirrored left-right                  0.445
  rotated 90 degrees                   0.117
  thicker strokes (brighter)           0.972
  light noise added                    0.958
(Mirroring and rotation genuinely change some digits, e.g. 6 and 9, so a human
 would also hesitate; inversion and brightness should not confuse a reader at all.)



A network learns y = x^2 from examples between -3 and 3:
   x     true y   network
 -2.0       4.0       4.1
  0.5       0.2       0.3
  2.5       6.2       6.4
  5.0      25.0      16.5
 10.0     100.0      35.5
 20.0     400.0      73.5
Inside the training range it is accurate; outside it the 'rule' was never learned.
A student who understands squaring extrapolates immediately.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Digit classifier accuracy (chance = 0.10):
  same distribution (new images)       0.984
  colours inverted (white on black)    0.000
  mirrored left-right                  0.445
  rotated 90 degrees                   0.117
  thicker strokes (brighter)           0.972
  light noise added                    0.958
(Mirroring and rotation genuinely change some digits, e.g. 6 and 9, so a human
 would also hesitate; inversion and brightness should not confuse a reader at all.)
 
A network learns y = x^2 from examples between -3 and 3:
   x     true y   network
 -2.0       4.0       4.1
  0.5       0.2       0.3
  2.5       6.2       6.4
  5.0      25.0      16.5
 10.0     100.0      35.5
 20.0     400.0      73.5
Inside the training range it is accurate; outside it the 'rule' was never learned.
A student who understands squaring extrapolates immediately.
```

**What the output shows**

A classifier that is 98 percent accurate on new images from its training distribution scores exactly 0 percent when colours are inverted, a change that would not trouble a human reader at all. The squaring network is accurate within its training range but badly wrong beyond it. Both illustrate the difference between fitting a distribution and grasping a concept.

Why does inversion destroy the classifier so completely? In the training images the background is always 0 (black) and the strokes are bright. The network learned that bright pixels in certain places indicate certain digits. After inversion almost every pixel is bright and the strokes are dark, a pattern unlike anything it has seen, so its outputs are effectively arbitrary. A person knows that the shape is what matters and the colour does not; the network was never told this and had no reason to discover it from the data it was given. What looks like a small change to us is, in the network's input space, a jump to a region it has never visited.

Pause and reflect: The squaring network was accurate on every example it could have been tested on during training. Only questions outside the training range exposed the problem. How would you know, for a real system, whether a new situation is inside or outside the range of its training? Who would be responsible for checking?

**Investigate**

1. Add inverted images to the training data. Does the classifier now handle both? Is that the same as understanding that colour does not matter?
2. Replace the squaring network with numpy.polyfit(x, x\*\*2, 2). Why does this model extrapolate perfectly, and what prior knowledge did choosing it build in?

## 4. Catastrophic forgetting

**File:** `l7_forgetting.py`

Neural networks trained on one task and then another tend to overwrite the first, a phenomenon first documented in connectionist models by McCloskey and Cohen, who called it catastrophic interference (McCloskey and Cohen, 1989), and by Ratcliff (Ratcliff, 1990), and addressed in modern deep learning by methods such as elastic weight consolidation (Kirkpatrick et al., 2017).

In [5]:
"""
Layer 7 example: catastrophic forgetting (McCloskey and Cohen, 1989;
Ratcliff, 1990; Kirkpatrick et al., 2017).
 
File: l7_forgetting.py
 
Purpose:
  People learn new skills without erasing old ones. Standard neural networks
  trained on task A and then on task B tend to OVERWRITE what they knew about
  A, because the same weights are adjusted for the new objective. Learning
  continually, like a person, is one of the abilities often listed as missing
  on the road to general intelligence.
 
  Task A: recognise digits 0-4.   Task B: recognise digits 5-9.
 
How it works:
  1. The digit images are split into training and test sets, and each set
     into task A (0-4) and task B (5-9).
  2. Sequential training: the network learns A, is tested on both tasks,
     then learns B only and is tested again.
  3. Rehearsal: the same, but 10% of task A's examples are replayed (mixed in)
     while learning B, a simple remedy.
  4. Joint training: all digits at once, the usual approach, for comparison.
  partial_fit() trains in mini-batches of 32, so we control exactly which
  examples the network sees in each phase.
 
What to look for in the output:
  - After learning B, accuracy on A falls from 0.987 to 0.000.
  - Replaying just 10% of A keeps 0.845 on A while still learning B (0.965).
  - Joint training does well on both, but requires all the data up front.
 
Requires: NumPy and scikit-learn (pip install numpy scikit-learn).
"""
 
import numpy as np
import warnings
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier
warnings.filterwarnings("ignore")      # hide convergence warnings for clarity
 
X, y = load_digits(return_X_y=True)
X = X / 16.0                           # scale pixel values to 0-1
rng = np.random.default_rng(1)
idx = rng.permutation(len(X)); tr, te = idx[:1300], idx[1300:]
# Split each set by label: task A is digits 0-4, task B is digits 5-9.
A_tr, B_tr = tr[y[tr] < 5], tr[y[tr] >= 5]
A_te, B_te = te[y[te] < 5], te[y[te] >= 5]
# partial_fit must be told all possible labels in advance.
classes = np.arange(10)
 
 
def acc(m, ids):
    # Accuracy of model m on the examples with the given indices.
    return (m.predict(X[ids]) == y[ids]).mean()
 
 
def train_phase(model, ids, epochs=60):
    # Trains only on the given examples, in shuffled mini-batches of 32.
    for _ in range(epochs):
        order = rng.permutation(ids)
        for s in range(0, len(order), 32):
            model.partial_fit(X[order[s:s+32]], y[order[s:s+32]], classes=classes)
 
 
print("SEQUENTIAL TRAINING (no rehearsal)")
m = MLPClassifier(hidden_layer_sizes=(64,), learning_rate_init=0.005, random_state=0)
train_phase(m, A_tr)
print(f"  after learning A: accuracy on A {acc(m, A_te):.3f}, on B {acc(m, B_te):.3f}")
train_phase(m, B_tr)
print(f"  after learning B: accuracy on A {acc(m, A_te):.3f},"
      f" on B {acc(m, B_te):.3f}   <- A forgotten")
 
print("\nSEQUENTIAL TRAINING WITH REHEARSAL (replay 10% of task A during task B)")
m2 = MLPClassifier(hidden_layer_sizes=(64,), learning_rate_init=0.005, random_state=0)
train_phase(m2, A_tr)
# Keep a random 10% of task A's training examples to mix into phase B.
replay = rng.choice(A_tr, size=len(A_tr) // 10, replace=False)
train_phase(m2, np.concatenate([B_tr, replay]))
print(f"  after learning B: accuracy on A {acc(m2, A_te):.3f}, on B"
      f" {acc(m2, B_te):.3f}")
 
print("\nJOINT TRAINING (all data at once, the usual approach)")
m3 = MLPClassifier(hidden_layer_sizes=(64,), learning_rate_init=0.005, random_state=0)
train_phase(m3, tr)
print(f"  accuracy on A {acc(m3, A_te):.3f}, on B {acc(m3, B_te):.3f}")
print("\nLarge models avoid this by training once on everything, then staying FIXED.")
print("Learning continuously from experience, as people do, remains an open problem.")

SEQUENTIAL TRAINING (no rehearsal)


  after learning A: accuracy on A 0.987, on B 0.000


  after learning B: accuracy on A 0.000, on B 0.977   <- A forgotten

SEQUENTIAL TRAINING WITH REHEARSAL (replay 10% of task A during task B)


  after learning B: accuracy on A 0.845, on B 0.965

JOINT TRAINING (all data at once, the usual approach)


  accuracy on A 0.975, on B 0.961

Large models avoid this by training once on everything, then staying FIXED.
Learning continuously from experience, as people do, remains an open problem.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
SEQUENTIAL TRAINING (no rehearsal)
  after learning A: accuracy on A 0.987, on B 0.000
  after learning B: accuracy on A 0.000, on B 0.977   <- A forgotten
 
SEQUENTIAL TRAINING WITH REHEARSAL (replay 10% of task A during task B)
  after learning B: accuracy on A 0.845, on B 0.965
 
JOINT TRAINING (all data at once, the usual approach)
  accuracy on A 0.975, on B 0.961
 
Large models avoid this by training once on everything, then staying FIXED.
Learning continuously from experience, as people do, remains an open problem.
```

**What the output shows**

After learning digits 5 to 9, the network's accuracy on digits 0 to 4 falls from 99 percent to zero. Replaying just 10 percent of the old data preserves most of it. Today's large models largely sidestep the problem by training once on a vast mixture of data and then remaining fixed, supplemented by retrieval and context; genuinely learning from ongoing experience in the way people do remains an active research area.

**Investigate**

1. Vary the rehearsal proportion (1, 5, 20 and 50 percent) and plot accuracy on task A against it.
2. Why might continual learning raise safety concerns as well as capability benefits for a deployed system?

## 5. Why forecasts disagree

**File:** `l7_forecasting.py`

In [6]:
"""
Layer 7 example: why forecasts about AGI disagree.
 
File: l7_forecasting.py
 
Purpose:
  People often forecast when AI will reach "human level" by drawing a trend
  through past benchmark scores. This program shows why such forecasts can
  disagree wildly even when they use exactly the same history. We fit several
  reasonable-looking curves to a SYNTHETIC score series (invented numbers for
  illustration, not real data) and ask when each one crosses a 'human level'
  of 90%.
 
How it works:
  1. An S-shaped (logistic) curve plus random noise creates eleven years of
     scores, 2016 to 2026.
  2. Three models are fitted to that history: a straight line through the last
     five years, exponential growth, and a logistic curve that levels off at 80.
  3. Each fitted curve is extended to 2060 to find the first year it reaches 90.
  4. The logistic model is refitted with different assumed ceilings, and the
     fit error is reported for each.
 
What to look for in the output:
  - The linear and exponential models both say 2027; the plateau model says
    "never".
  - Ceilings of 85, 92, 100 and 120 fit the history almost equally well
    (errors 3.1 to 3.5 points) but give forecasts from 2027 to "never".
    The data cannot tell us the ceiling, yet the forecast depends on it.
 
Requires: NumPy (pip install numpy).
"""
 
import numpy as np
 
# Fixed seed so the synthetic history is identical every run.
rng = np.random.default_rng(3)
years = np.arange(2016, 2027)
# Synthetic history: an S-shaped curve centred on 2024, with noise added
true = 100 / (1 + np.exp(-0.55 * (years - 2024)))
scores = np.clip(true + rng.normal(0, 2.5, len(years)), 0, 100)
print("Synthetic score history:", dict(zip(years.tolist(), scores.round(1).tolist())))
TARGET = 90.0                      # the 'human level' threshold we forecast
future = np.arange(2027, 2061)     # years to search for the first crossing
 
# Model 1: straight line through the last five years
p1 = np.polyfit(years[-5:], scores[-5:], 1)
# Model 2: exponential growth fitted to all years (a straight line in log space)
p2 = np.polyfit(years, np.log(np.maximum(scores, 0.5)), 1)
 
 
# Model 3: logistic curve that saturates at a ceiling of 80 (a 'plateau' view).
# k controls how steep the S is; t0 is the year of fastest growth.
def logistic(t, k, t0, cap=80): return cap / (1 + np.exp(-k * (t - t0)))
 
 
# Fit k and t0 by brute-force grid search: try 60 x 120 combinations and keep
# the one with the smallest squared error against the history.
best = min(((k, t0) for k in np.linspace(0.1, 1.5, 60)
            for t0 in np.linspace(2018, 2035, 120)),
           key=lambda kt: ((logistic(years, *kt) - scores) ** 2).sum())
 
 
def first_crossing(values):
    # First future year whose value reaches TARGET, or None if it never does.
    hit = np.where(values >= TARGET)[0]
    return int(future[hit[0]]) if len(hit) else None
 
 
forecasts = {
    "linear trend (last 5 years)": first_crossing(np.polyval(p1, future)),
    "exponential growth": first_crossing(np.exp(np.polyval(p2, future))),
    "logistic with plateau at 80": first_crossing(logistic(future, *best)),
}
for name, yr in forecasts.items():
    print(f"  {name:30s} -> reaches {TARGET:.0f}: {yr if yr else 'never'}")
 
# Sensitivity: the plateau model depends entirely on where we assume the ceiling is
print("\nLogistic model with different assumed ceilings (all fit"
      " the history similarly):")
for cap in (85, 92, 100, 120):
    fit = min(((k, t0) for k in np.linspace(0.1, 1.5, 60)
               for t0 in np.linspace(2018, 2035, 120)),
              key=lambda kt: ((logistic(years, *kt, cap=cap) - scores) ** 2).sum())
    # Root-mean-square error: the typical gap, in points, between fit and history
    err = np.sqrt(((logistic(years, *fit, cap=cap) - scores) ** 2).mean())
    yr = first_crossing(logistic(future, *fit, cap=cap))
    print(f"  ceiling {cap:3d}: fit error {err:4.1f} points"
          f" -> reaches {TARGET:.0f}: {yr if yr else 'never'}")
print("The history cannot tell us the ceiling, yet the forecast depends on it.")
print("Treat confident single-date predictions with caution.")

Synthetic score history: {2016: 6.3, 2017: 0.0, 2018: 4.6, 2019: 4.6, 2020: 8.8, 2021: 15.6, 2022: 19.9, 2023: 36.0, 2024: 47.8, 2025: 71.7, 2026: 75.6}
  linear trend (last 5 years)    -> reaches 90: 2027
  exponential growth             -> reaches 90: 2027
  logistic with plateau at 80    -> reaches 90: never

Logistic model with different assumed ceilings (all fit the history similarly):


  ceiling  85: fit error  3.5 points -> reaches 90: never
  ceiling  92: fit error  3.2 points -> reaches 90: 2030


  ceiling 100: fit error  3.1 points -> reaches 90: 2028
  ceiling 120: fit error  3.2 points -> reaches 90: 2027
The history cannot tell us the ceiling, yet the forecast depends on it.
Treat confident single-date predictions with caution.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Synthetic score history: {2016: 6.3, 2017: 0.0, 2018: 4.6, 2019: 4.6, 2020: 8.8, 2021: 15.6, 2022: 19.9, 2023: 36.0, 2024: 47.8, 2025: 71.7, 2026: 75.6}
  linear trend (last 5 years)    -> reaches 90: 2027
  exponential growth             -> reaches 90: 2027
  logistic with plateau at 80    -> reaches 90: never
 
Logistic model with different assumed ceilings (all fit the history similarly):
  ceiling  85: fit error  3.5 points -> reaches 90: never
  ceiling  92: fit error  3.2 points -> reaches 90: 2030
  ceiling 100: fit error  3.1 points -> reaches 90: 2028
  ceiling 120: fit error  3.2 points -> reaches 90: 2027
The history cannot tell us the ceiling, yet the forecast depends on it.
Treat confident single-date predictions with caution.
```

**What the output shows**

The data in this program is invented for illustration. The point is general: curves with different long-run behaviour can fit a short history almost equally well, so the history alone cannot settle the question. Real forecasting faces the additional problem that there is no agreed measure of general intelligence to plot.

**Investigate**

1. Add three more synthetic years that show slowing growth (for example 78, 79, 80). How do the forecasts change?
2. Find a published forecast about AI and identify the assumptions it depends on. Which could be tested in the next two years?

## 6. Specification gaming

**File:** `l7_specification_gaming.py`

In [7]:
"""
Layer 7 example: specification gaming, a core alignment problem.
 
File: l7_specification_gaming.py
 
Purpose:
  We WANT an agent to finish a race along a track. We REWARD it for collecting
  checkpoints, assuming that encourages progress. But one checkpoint respawns,
  so a reward-maximising agent may learn to circle it forever instead of
  finishing. This mirrors a real case reported by OpenAI in a boat-racing
  game, discussed by Amodei et al. (2016) and catalogued with many others by
  Krakovna et al. (2020). The agent does exactly what it is rewarded for, not
  what we meant.
 
How it works:
  1. The track has positions 0 to 7; position 7 is the finish. The agent can
     move forward or back.
  2. Two reward designs are compared:
       checkpoints : +3 every time the agent enters position 2, +5 once for
                     finishing
       finish      : +10 for finishing, -0.1 for every other step
  3. Tabular Q-learning (Chapter 3) trains an agent under each design for
     3,000 episodes, exploring less as it learns (epsilon decays).
  4. The trained agent then acts greedily for up to 40 steps and we record
     its path and total reward.
 
What to look for in the output:
  - Under "checkpoints" the agent shuttles between positions 2 and 3,
    collecting 60 points in 40 steps, and never finishes.
  - Under "finish" it drives straight to the line in 7 steps.
  Nothing malfunctioned: looping really does earn more reward.
 
Requires: NumPy (pip install numpy).
"""
 
import numpy as np
 
rng = np.random.default_rng(0)
TRACK = 8                    # positions 0..7; position 7 is the finish line
# a checkpoint here respawns every time you leave and return
LOOP = 2
ACTIONS = ["forward", "back"]
 
 
def step(pos, action, reward_design):
    # Moves the agent one position and returns (new position, reward, finished?).
    nxt = min(pos + 1, TRACK - 1) if action == 0 else max(pos - 1, 0)
    done = nxt == TRACK - 1
    if reward_design == "checkpoints":
        r = 3.0 if nxt == LOOP else 0.0            # respawning checkpoint worth points
        # finishing is worth a little more, ONCE
        r += 5.0 if done else 0.0
    else:
        # "finish": reward only the outcome we want
        r = 10.0 if done else -0.1
    return nxt, r, done
 
 
def train(reward_design, episodes=3000, max_steps=40, gamma=0.95):
    # Q[pos, action] estimates the long-term reward of taking an action there.
    Q = np.zeros((TRACK, 2))
    eps = 1.0                                      # start by exploring at random
    for ep in range(episodes):
        pos, done, t = 0, False, 0
        while not done and t < max_steps:
            # Epsilon-greedy: explore with probability eps, otherwise exploit.
            a = int(rng.integers(2)) if rng.random() < eps else int(Q[pos].argmax())
            nxt, r, done = step(pos, a, reward_design)
            # Q-learning update: move the estimate towards reward + discounted
            # value of the best next action (learning rate 0.1).
            Q[pos, a] += 0.1 * (r + (0 if done else gamma * Q[nxt].max()) - Q[pos, a])
            pos, t = nxt, t + 1
        eps = max(0.02, eps * 0.998)               # explore a little less each episode
    return Q
 
 
def rollout(Q, reward_design, max_steps=40):
    # Runs the trained agent greedily and records what it actually does.
    pos, done, t, total, path = 0, False, 0, 0.0, [0]
    while not done and t < max_steps:
        pos, r, done = step(pos, int(Q[pos].argmax()), reward_design)
        total += r; t += 1; path.append(pos)
    return done, total, path
 
 
for design in ("checkpoints", "finish"):
    Q = train(design)
    finished, total, path = rollout(Q, design)
    print(f"Reward design: {design}")
    print(f"  finished race: {finished}   reward collected: {total:.1f}"
          f"   steps: {len(path) - 1}")
    print(f"  path: {path[:20]}{' ...' if len(path) > 20 else ''}\n")
 
print("With checkpoint rewards the agent earns MORE reward by"
      " looping and never finishes.")
print("Designing objectives that capture what we really want gets"
      " harder as systems become")
print("more capable at finding unexpected ways to score highly.")

Reward design: checkpoints
  finished race: False   reward collected: 60.0   steps: 40
  path: [0, 1, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3] ...



Reward design: finish
  finished race: True   reward collected: 9.4   steps: 7
  path: [0, 1, 2, 3, 4, 5, 6, 7]

With checkpoint rewards the agent earns MORE reward by looping and never finishes.
Designing objectives that capture what we really want gets harder as systems become
more capable at finding unexpected ways to score highly.


**Output shown in the guide** (timings, and anything drawn from live services, may differ on your machine):

```text
Reward design: checkpoints
  finished race: False   reward collected: 60.0   steps: 40
  path: [0, 1, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3] ...
 
Reward design: finish
  finished race: True   reward collected: 9.4   steps: 7
  path: [0, 1, 2, 3, 4, 5, 6, 7]
 
With checkpoint rewards the agent earns MORE reward by looping and never finishes.
Designing objectives that capture what we really want gets harder as systems become
more capable at finding unexpected ways to score highly.
```

**What the output shows**

The agent rewarded for checkpoints collects 60 points by circling forever and never finishes; the agent rewarded only for finishing takes the direct route. Nothing malfunctioned. The agent optimised exactly what it was given. As systems become more capable of discovering unexpected strategies, stating the right objective, and checking that a system has actually learned it, become more important.

**Investigate**

1. Change the finishing reward from 5 to 50 while keeping checkpoints. At what value does the agent choose to finish?
2. Describe a realistic workplace example where rewarding a measurable proxy (for example, tickets closed) could produce specification gaming by people or by an AI agent.